In [7]:
import glob, zarr, os, napari
import dask.array as da
from tqdm.auto import tqdm
from natsort import natsorted
import napari
import pandas as pd
SCALE_TUPLE = (0.165, 0.165)
SPLIT_IMAGE_KWS =  ['top', 'bot', 'left', 'right']

# Generating full image FOVs

In [8]:
zarr_fns = glob.glob('/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep*/mouse_*/zarr/*max_proj.zarr')
# Filter and sort
zarr_fns = natsorted([
                        fn for fn in zarr_fns
                        if not any(x in fn for x in SPLIT_IMAGE_KWS)])
zarr_fns

['/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep1/mouse_1/zarr/rep_1_mouse_1_max_proj.zarr',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep1/mouse_2/zarr/rep_1_mouse_2_max_proj.zarr',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep1/mouse_3/zarr/rep_1_mouse_3_max_proj.zarr',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep1/mouse_4/zarr/rep_1_mouse_4_max_proj.zarr',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep1/mouse_5/zarr/rep_1_mouse_5_max_proj.zarr',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_1/zarr/rep_2_mouse_1_max_proj.zarr',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_2/zarr/rep_2_mouse_2_max_proj.zarr',
 '/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep2/mouse_7/zarr/rep_2_mouse_7_max_proj.zarr',
 '/mnt/O

In [12]:
df_metadata = pd.read_pickle('/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/results/arx/metadata.pkl')

In [13]:
df_metadata

,mouse_num,replicate,condition,basename_stem
0,1,1,H2O,20250815_40X_TimerMtb_BP_mice1_DAPI_TimerG_Tim...
1,2,1,DMSO,20250901_40X_TimerMtb_BP_mice2_mice3_mice4_DAP...
2,3,1,RIF,20250901_40X_TimerMtb_BP_mice2_mice3_mice4_DAP...
3,4,1,RIF,20250901_40X_TimerMtb_BP_mice2_mice3_mice4_DAP...
4,5,1,RIF,20250901_40X_TimerMtb_BP_mice5_mice6_mice7_DAP...
5,6,1,RIF,20250901_40X_TimerMtb_BP_mice5_mice6_mice7_DAP...
6,7,1,PZA,20250901_40X_TimerMtb_BP_mice5_mice6_mice7_DAP...
7,8,1,PZA,20250821_40X_TimerMtb_BP_mice8_mice9_DAPI_Time...
8,9,1,PZA,20250821_40X_TimerMtb_BP_mice8_mice9_DAPI_Time...
9,10,1,PZA,20250814_40X_TimerMtb_BP_mice10_DAPI_TimerG_Ti...


In [18]:
import napari
import glob
import os
import json
import re
import dask.array as da
import pandas as pd
from tqdm.auto import tqdm
from pathlib import Path

# Setup QC directory
qc_dir = Path("/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/results/screenshots_v2")
qc_dir.mkdir(exist_ok=True)

mtb_colormap = {1: '#FFFF00'} # yellow: #FFFF00

# Regex to parse info from filename
rep_pattern = re.compile(r"rep_(\d+)")
mouse_pattern = re.compile(r"mouse_(\d+)")

for zarr_fn in tqdm(zarr_fns):
    # --- Parse Identity ---
    basename = os.path.basename(os.path.normpath(zarr_fn))
    
    rep_match = rep_pattern.search(basename)
    rep_id = int(rep_match.group(1)) if rep_match else None
    
    mouse_match = mouse_pattern.search(basename)
    mouse_id = int(mouse_match.group(1)) if mouse_match else None
    
    # --- Lookup Condition ---
    condition = "Unknown"
    if rep_id is not None and mouse_id is not None:
        # Query the metadata df for the matching row
        # keys: mouse_num, replicate, condition
        match = df_metadata[
            (df_metadata['mouse_num'] == mouse_id) & 
            (df_metadata['replicate'] == rep_id)
        ]
        if not match.empty:
            condition = match.iloc[0]['condition']

    # --- Load Data ---
    pyramid_image_stack = [da.from_zarr(fn) for fn in glob.glob(f"{zarr_fn}/s*")]
    cyto_masks = da.from_zarr(f'{zarr_fn}/labels/cyto_seg/0')
    mtb_masks = da.from_zarr(f'{zarr_fn}/labels/ground_truth_mtb/0')

    # --- Viewer Setup ---
    viewer = napari.Viewer(title=basename)
    
    viewer.add_image(pyramid_image_stack, channel_axis=0, 
                     colormap=['blue', 'green', 'magenta'], scale=SCALE_TUPLE)
    viewer.add_labels(cyto_masks, name='cytoplasm', scale=SCALE_TUPLE,)# contour=4)
    viewer.add_labels(mtb_masks, name='mtb', scale=SCALE_TUPLE, 
                      colormap=mtb_colormap,) #contour=4)

    viewer.scale_bar.visible = True
    viewer.scale_bar.unit = "um"

    # --- Screenshot Keybinding ---
    @viewer.bind_key('s')
    def save_qc_screenshot(viewer):
        # Define base pattern
        base_name_str = f"{basename}_{condition}_QC"
        
        # Check for existence and increment counter if needed
        counter = 0
        candidate_name = base_name_str
        while (qc_dir / f"{candidate_name}.png").exists():
            counter += 1
            candidate_name = f"{base_name_str}_{counter}"
        
        # 1. Save Image
        image_path = qc_dir / f"{candidate_name}.png"
        viewer.screenshot(path=str(image_path))
        
        # 2. Save Metadata
        meta = {
            "file": basename,
            "condition": condition,
            "mouse": mouse_id,
            "rep": rep_id,
            "zoom": viewer.camera.zoom,
            "center": list(viewer.camera.center),
            "layers": {l.name: l.visible for l in viewer.layers},
            "save_iteration": counter
        }
        
        json_path = qc_dir / f"{candidate_name}.json"
        with open(json_path, 'w') as f:
            json.dump(meta, f, indent=4)
            
        viewer.status = f"Saved: {candidate_name}"
        print(f"Captured: {candidate_name}")

    viewer.show(block=True)

  0%|          | 0/9 [00:00<?, ?it/s]

/home/dayn/miniconda3/envs/macrohet_fixed/lib/python3.10/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (30089, 49350) exceeds GL_MAX_TEXTURE_SIZE 32768 in at least one axis and will be downsampled. Rendering is currently in 2D mode.
  warnings.warn(
/home/dayn/miniconda3/envs/macrohet_fixed/lib/python3.10/site-packages/napari/utils/colormaps/colormap.py:455: UserWarning: color_dict did not provide a default color. Missing keys will be transparent. To provide a default color, use the key `None`, or provide a defaultdict instance.
  warn(
/home/dayn/miniconda3/envs/macrohet_fixed/lib/python3.10/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (31334, 49997) exceeds GL_MAX_TEXTURE_SIZE 32768 in at least one axis and will be downsampled. Rendering is currently in 2D mode.
  warnings.warn(


Captured: rep_1_mouse_1_max_proj.zarr_H2O_QC
Captured: rep_1_mouse_1_max_proj.zarr_H2O_QC_1
Captured: rep_1_mouse_1_max_proj.zarr_H2O_QC_2
Captured: rep_1_mouse_1_max_proj.zarr_H2O_QC_3


/home/dayn/miniconda3/envs/macrohet_fixed/lib/python3.10/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (30089, 49350) exceeds GL_MAX_TEXTURE_SIZE 32768 in at least one axis and will be downsampled. Rendering is currently in 2D mode.
  warnings.warn(


Captured: rep_1_mouse_1_max_proj.zarr_H2O_QC_4
Captured: rep_1_mouse_1_max_proj.zarr_H2O_QC_5
Captured: rep_1_mouse_1_max_proj.zarr_H2O_QC_6
Captured: rep_1_mouse_1_max_proj.zarr_H2O_QC_7
Captured: rep_1_mouse_1_max_proj.zarr_H2O_QC_8
Captured: rep_1_mouse_1_max_proj.zarr_H2O_QC_9
Captured: rep_1_mouse_1_max_proj.zarr_H2O_QC_10
Captured: rep_1_mouse_1_max_proj.zarr_H2O_QC_11


...

Captured: rep_2_mouse_7_max_proj.zarr_RIF_QC
Captured: rep_2_mouse_7_max_proj.zarr_RIF_QC_1
Captured: rep_2_mouse_7_max_proj.zarr_RIF_QC_2
Captured: rep_2_mouse_7_max_proj.zarr_RIF_QC_3
Captured: rep_2_mouse_7_max_proj.zarr_RIF_QC_4
Captured: rep_2_mouse_7_max_proj.zarr_RIF_QC_5
Captured: rep_2_mouse_7_max_proj.zarr_RIF_QC_6
Captured: rep_2_mouse_7_max_proj.zarr_RIF_QC_7
Captured: rep_2_mouse_7_max_proj.zarr_RIF_QC_8
Captured: rep_2_mouse_7_max_proj.zarr_RIF_QC_9
Captured: rep_2_mouse_7_max_proj.zarr_RIF_QC_10
Captured: rep_2_mouse_7_max_proj.zarr_RIF_QC_11
Captured: rep_2_mouse_7_max_proj.zarr_RIF_QC_12
Captured: rep_2_mouse_7_max_proj.zarr_RIF_QC_13
Captured: rep_2_mouse_7_max_proj.zarr_RIF_QC_14
Captured: rep_2_mouse_7_max_proj.zarr_RIF_QC_15
Captured: rep_2_mouse_7_max_proj.zarr_RIF_QC_16
Captured: rep_2_mouse_7_max_proj.zarr_RIF_QC_17
Captured: rep_2_mouse_7_max_proj.zarr_RIF_QC_18
Captured: rep_2_mouse_7_max_proj.zarr_RIF_QC_19
Captured: rep_2_mouse_7_max_proj.zarr_RIF_QC_20
Capt

/home/dayn/miniconda3/envs/macrohet_fixed/lib/python3.10/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (38534, 52070) exceeds GL_MAX_TEXTURE_SIZE 32768 in at least one axis and will be downsampled. Rendering is currently in 2D mode.
  warnings.warn(
/home/dayn/miniconda3/envs/macrohet_fixed/lib/python3.10/site-packages/napari/utils/colormaps/colormap.py:455: UserWarning: color_dict did not provide a default color. Missing keys will be transparent. To provide a default color, use the key `None`, or provide a defaultdict instance.
  warn(
/home/dayn/miniconda3/envs/macrohet_fixed/lib/python3.10/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (39629, 52070) exceeds GL_MAX_TEXTURE_SIZE 32768 in at least one axis and will be downsampled. Rendering is currently in 2D mode.
  warnings.warn(


Captured: rep_2_mouse_8_max_proj.zarr_RIF_QC
Captured: rep_2_mouse_8_max_proj.zarr_RIF_QC_1
Captured: rep_2_mouse_8_max_proj.zarr_RIF_QC_2
Captured: rep_2_mouse_8_max_proj.zarr_RIF_QC_3
Captured: rep_2_mouse_8_max_proj.zarr_RIF_QC_4
Captured: rep_2_mouse_8_max_proj.zarr_RIF_QC_5


/home/dayn/miniconda3/envs/macrohet_fixed/lib/python3.10/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (38534, 52070) exceeds GL_MAX_TEXTURE_SIZE 32768 in at least one axis and will be downsampled. Rendering is currently in 2D mode.
  warnings.warn(


Captured: rep_2_mouse_8_max_proj.zarr_RIF_QC_6
Captured: rep_2_mouse_8_max_proj.zarr_RIF_QC_7
Captured: rep_2_mouse_8_max_proj.zarr_RIF_QC_8
Captured: rep_2_mouse_8_max_proj.zarr_RIF_QC_9
Captured: rep_2_mouse_8_max_proj.zarr_RIF_QC_10
Captured: rep_2_mouse_8_max_proj.zarr_RIF_QC_11
Captured: rep_2_mouse_8_max_proj.zarr_RIF_QC_12
Captured: rep_2_mouse_8_max_proj.zarr_RIF_QC_13
Captured: rep_2_mouse_8_max_proj.zarr_RIF_QC_14
Captured: rep_2_mouse_8_max_proj.zarr_RIF_QC_15
Captured: rep_2_mouse_8_max_proj.zarr_RIF_QC_16
Captured: rep_2_mouse_8_max_proj.zarr_RIF_QC_17
Captured: rep_2_mouse_8_max_proj.zarr_RIF_QC_18
Captured: rep_2_mouse_8_max_proj.zarr_RIF_QC_19


### started at 1705

In [19]:
from datetime import datetime

# Get and print current time
print(datetime.now())

# Optional: Print just the time (HH:MM:SS)
print(datetime.now().strftime("%H:%M:%S"))

2025-12-22 11:22:02.779232
11:22:02


In [20]:
print()

### Iterating over mice images to check shape

In [3]:
fns = natsorted(glob.glob('/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep*/mouse_*/zarr/*.zarr/0/4'))
total=0
for fn in tqdm(fns):
    print(fn.split('whole_slice/')[1].split('/zarr')[0])
    print(da.from_zarr(fn).shape)
    # max_proj_3ch_image = da.from_zarr(fn).max(axis=2).compute()

  0%|          | 0/27 [00:00<?, ?it/s]

rep1/mouse_1
(1, 3, 11, 1958, 3124)
rep1/mouse_2
(1, 3, 11, 2347, 3124)
rep1/mouse_3
(1, 3, 11, 3643, 2347)
rep1/mouse_4
(1, 3, 11, 1569, 792)
rep1/mouse_5
(1, 3, 11, 2476, 3384)
rep1/mouse_6
(1, 3, 11, 3384, 2865)
rep1/mouse_7
(1, 3, 11, 3384, 2995)
rep1/mouse_8
(1, 3, 11, 2606, 3513)
rep1/mouse_9
(1, 3, 11, 3772, 2736)
rep1/mouse_10
(1, 3, 11, 2606, 3643)
rep1/mouse_11
(1, 3, 11, 2606, 3772)
rep2/mouse_1
(1, 3, 17, 2865, 3254)
rep2/mouse_2
(1, 3, 17, 3124, 2088)
rep2/mouse_3
(1, 3, 17, 2736, 2217)
rep2/mouse_4
(1, 3, 15, 3902, 2736)
rep2/mouse_5
(1, 3, 15, 3513, 2217)
rep2/mouse_6
(1, 3, 13, 3513, 2865)
rep2/mouse_7
(1, 3, 11, 2476, 3124)
rep2/mouse_8
(1, 3, 11, 2476, 3254)
rep2/mouse_9
(1, 3, 11, 3254, 2736)
rep2/mouse_10
(1, 3, 13, 4032, 3124)
rep2/mouse_11
(1, 3, 13, 3513, 2865)
rep2/mouse_12
(1, 3, 17, 3384, 2865)
rep2/mouse_13
(1, 3, 19, 3254, 2736)
rep2/mouse_14
(1, 3, 19, 3254, 2736)
rep2/mouse_15
(1, 3, 13, 3254, 2865)
rep2/mouse_16
(1, 3, 15, 3513, 2865)


In [3]:
import numpy as np

In [44]:
%%time
test = da.from_zarr(fn).max(axis=2).compute()

CPU times: user 697 ms, sys: 554 ms, total: 1.25 s
Wall time: 2.83 s


In [48]:
%%time
test = da.from_zarr(fn).max(axis=2).compute()

CPU times: user 2.38 s, sys: 3.4 s, total: 5.78 s
Wall time: 24.1 s


In [49]:
viewer = napari.Viewer(title='mosaic 4 scale')
viewer.add_image(test,channel_axis=1)

[<Image layer 'Image' at 0x76264a077790>,
 <Image layer 'Image [1]' at 0x76264a096530>,
 <Image layer 'Image [2]' at 0x76264a0393c0>]

In [32]:
'/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep1/mouse_10/zarr/20250814_40X_TimerMtb_BP_mice10_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250814_5375.zarr/0/'

'2.29e+12'

# Generating the mosaic

In [7]:
fns = glob.glob('/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep*/mouse_*/zarr/*.zarr/0/4')
maxy, maxx = 0,0
for fn in tqdm(fns):
    y, x = da.from_zarr(fn).shape[-2],da.from_zarr(fn).shape[-1]
    if y > maxy:
        maxy = y
    if x > maxx:
        maxx = x
    # print(y,x)
    # max_proj_3ch_image = da.from_zarr(fn).max(axis=2).compute()

  0%|          | 0/27 [00:00<?, ?it/s]

In [78]:
da.from_zarr(fn).max(axis=2)[0]

dask.array<getitem, shape=(3, 2865, 3254), dtype=>u2, chunksize=(1, 1024, 1024), chunktype=numpy.ndarray>

In [86]:
image

dask.array<getitem, shape=(3, 2865, 3254), dtype=>u2, chunksize=(1, 1024, 1024), chunktype=numpy.ndarray>

In [8]:
maxx

3772

In [89]:
da.pad(image, ((0,0), (0,dy), (0,dx)))

dask.array<concatenate, shape=(3, 4032, 3772), dtype=uint16, chunksize=(1, 1024, 1024), chunktype=numpy.ndarray>

In [84]:
dy = maxy - image.shape[-2]
dx = maxx - image.shape[-1]
image = da.pad(image, (0,dy, dx))

TypeError: `pad_value` must be composed of integral typed values.

In [85]:
dx, dy

(518, 1167)

In [9]:
stack = da.zeros((len(fns), 3, maxy, maxx))
for i, fn in tqdm(enumerate(fns), total = len(fns)):
    image = da.from_zarr(fn).max(axis=2)[0]#.compute()[0]
    dy = maxy - image.shape[-2]
    dx = maxx - image.shape[-1]
    image = da.pad(image, ((0,0), (0,dy), (0,dx)))
    image
    stack[i] = image

  0%|          | 0/27 [00:00<?, ?it/s]

In [19]:
stack

dask.array<setitem, shape=(27, 3, 4032, 3772), dtype=float64, chunksize=(27, 3, 455, 455), chunktype=numpy.ndarray>

In [25]:
import pandas as pd

In [27]:
metadata

,mouse_num,replicate,condition,basename_stem
0,1,1,H2O,20250815_40X_TimerMtb_BP_mice1_DAPI_TimerG_Tim...
1,2,1,DMSO,20250901_40X_TimerMtb_BP_mice2_mice3_mice4_DAP...
2,3,1,RIF,20250901_40X_TimerMtb_BP_mice2_mice3_mice4_DAP...
3,4,1,RIF,20250901_40X_TimerMtb_BP_mice2_mice3_mice4_DAP...
4,5,1,RIF,20250901_40X_TimerMtb_BP_mice5_mice6_mice7_DAP...
5,6,1,RIF,20250901_40X_TimerMtb_BP_mice5_mice6_mice7_DAP...
6,7,1,PZA,20250901_40X_TimerMtb_BP_mice5_mice6_mice7_DAP...
7,8,1,PZA,20250821_40X_TimerMtb_BP_mice8_mice9_DAPI_Time...
8,9,1,PZA,20250821_40X_TimerMtb_BP_mice8_mice9_DAPI_Time...
9,10,1,PZA,20250814_40X_TimerMtb_BP_mice10_DAPI_TimerG_Ti...


In [29]:
import numpy as np
import os
from skimage import io, exposure, img_as_ubyte
from tqdm import tqdm # Assuming tqdm is installed

# --- 1. Define Condition-Integrated Identifiers and Final Output Directory ---

# This list matches the EXACT required frame order (rep2/1-16, rep1/11, rep1/1-10) 
# and integrates the correct 'Treatment' from the latest metadata.
identifiers_with_condition = [
    "rep2_mouse_1_H2O", "rep2_mouse_2_H2O", "rep2_mouse_3_H2O", 
    "rep2_mouse_4_DMSO", "rep2_mouse_5_DMSO", "rep2_mouse_6_DMSO", 
    "rep2_mouse_7_RIF", "rep2_mouse_8_RIF", "rep2_mouse_9_RIF", 
    "rep2_mouse_10_RIF", "rep2_mouse_11_RIF", 
    "rep2_mouse_12_PZA", "rep2_mouse_13_PZA", "rep2_mouse_14_PZA", 
    "rep2_mouse_15_PZA", "rep2_mouse_16_PZA", 
    "rep1_mouse_11_DMSO", # Start of Rep1 block
    "rep1_mouse_1_H2O", "rep1_mouse_2_DMSO", "rep1_mouse_3_RIF", 
    "rep1_mouse_4_RIF", "rep1_mouse_5_RIF", "rep1_mouse_6_RIF", 
    "rep1_mouse_7_PZA", "rep1_mouse_8_PZA", "rep1_mouse_9_PZA", 
    "rep1_mouse_10_PZA"
]

output_root = "/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/results"
output_dir = os.path.join(output_root, "processed_frames")
os.makedirs(output_dir, exist_ok=True)

# --- 2. Setup (Ensure 'loaded_stack' is present) ---
# Assuming 'loaded_stack' of shape (27, 3, 4032, 3772) is in memory.
if 'loaded_stack' not in locals():
    print("WARNING: 'loaded_stack' not found. Ensure your data is loaded before running the loop.")
    # If testing:
    # loaded_stack = np.random.rand(len(identifiers_with_condition), 3, 100, 100).astype(np.float32)

# Check for length consistency
stack_size = loaded_stack.shape[0] if 'loaded_stack' in locals() else len(identifiers_with_condition)
if stack_size != len(identifiers_with_condition):
    raise ValueError(
        f"The time dimension of loaded_stack ({stack_size}) "
        f"does not match the number of identifiers ({len(identifiers_with_condition)})."
    )

# --- 3. Processing Loop ---

# Iterate directly through the time/Z dimension (Dim 0)
for i in tqdm(range(stack_size)):
    # Get the correct file identifier for the current frame, including the treatment
    file_identifier = identifiers_with_condition[i] 

    # Extract channels (Dim 1: 0=DAPI, 1=Green, 2=Magenta)
    dapi = loaded_stack[i, 0].copy()
    green = loaded_stack[i, 1].copy()
    magenta = loaded_stack[i, 2].copy()

    # Define contrast limits automatically (2nd and 98th percentiles)
    
    # DAPI
    p2_d, p98_d = np.percentile(dapi, (2, 98))
    dapi_rescaled = exposure.rescale_intensity(dapi, in_range=(p2_d, p98_d))
    
    # Green
    p2_g, p98_g = np.percentile(green, (2, 98))
    green_rescaled = exposure.rescale_intensity(green, in_range=(p2_g, p98_g))
    
    # Magenta
    p2_m, p98_m = np.percentile(magenta, (2, 98))
    magenta_rescaled = exposure.rescale_intensity(magenta, in_range=(p2_m, p98_m))

    # Create RGB Composite (all rescaled channels are float 0.0 to 1.0)
    
    r_channel = magenta_rescaled  # R channel: Magenta signal (Mtb Timer Red)
    g_channel = green_rescaled    # G channel: Green signal (Mtb Timer Green)
    # B channel: DAPI + Magenta signal, clipped to 1.0 (for DAPI/Magenta co-localization)
    b_channel = np.clip(dapi_rescaled + magenta_rescaled, 0.0, 1.0)
    
    # Stack into (Height, Width, 3)
    rgb_composite = np.dstack((r_channel, g_channel, b_channel))

    # Convert to 8-bit unsigned integer (0-255) for PNG saving
    # 
    rgb_uint8 = img_as_ubyte(rgb_composite)

    # Save output using the new file_identifier with the treatment
    filename = os.path.join(output_dir, f"{file_identifier}.png")
    io.imsave(filename, rgb_uint8)
    
    print(f"Frame {i} saved to {filename}")

print("Processing complete.")

  4%|████▎                                                                                                              | 1/27 [00:11<05:03, 11.67s/it]

Frame 0 saved to /mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/results/processed_frames/rep2_mouse_1_H2O.png


  7%|████████▌                                                                                                          | 2/27 [00:18<03:41,  8.88s/it]

Frame 1 saved to /mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/results/processed_frames/rep2_mouse_2_H2O.png


 11%|████████████▊                                                                                                      | 3/27 [00:24<02:55,  7.33s/it]

...

Frame 24 saved to /mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/results/processed_frames/rep1_mouse_8_PZA.png


 96%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████▊    | 26/27 [02:48<00:06,  6.51s/it]

Frame 25 saved to /mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/results/processed_frames/rep1_mouse_9_PZA.png


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 27/27 [02:55<00:00,  6.49s/it]

Frame 26 saved to /mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/results/processed_frames/rep1_mouse_10_PZA.png
Processing complete.


In [20]:
import numpy as np
import os
from skimage import io, exposure, img_as_ubyte

# Assumption: loaded_stack is already in memory with shape (27, 3, 4032, 3772)
# If testing, uncomment the line below to generate a dummy stack
# loaded_stack = np.random.rand(27, 3, 4032, 3772).astype(np.float32)

output_dir = "processed_frames"
os.makedirs(output_dir, exist_ok=True)

# Iterate directly through the time/Z dimension (Dim 0)
for i in tqdm(range(loaded_stack.shape[0])):
    # Extract channels (Dim 1: 0=DAPI, 1=Green, 2=Magenta)
    # Using 'copy()' to ensure we don't modify the original stack in memory
    dapi = loaded_stack[i, 0].copy()
    green = loaded_stack[i, 1].copy()
    magenta = loaded_stack[i, 2].copy()

    # Define contrast limits automatically (2nd and 98th percentiles)
    # DAPI
    p2_d, p98_d = np.percentile(dapi, (2, 98))
    dapi_rescaled = exposure.rescale_intensity(dapi, in_range=(p2_d, p98_d))
    
    # Green
    p2_g, p98_g = np.percentile(green, (2, 98))
    green_rescaled = exposure.rescale_intensity(green, in_range=(p2_g, p98_g))
    
    # Magenta
    p2_m, p98_m = np.percentile(magenta, (2, 98))
    magenta_rescaled = exposure.rescale_intensity(magenta, in_range=(p2_m, p98_m))

    # Create RGB Composite
    # R channel: Receives Magenta signal
    # G channel: Receives Green signal
    # B channel: Receives DAPI signal + Magenta signal (to create visual Magenta)
    
    # Initialize RGB arrays (float for calculation)
    r_channel = magenta_rescaled
    g_channel = green_rescaled
    # Add DAPI and Magenta for Blue channel, clip to avoid overflow
    b_channel = np.clip(dapi_rescaled + magenta_rescaled, 0, 1.0)
    if dapi_rescaled.dtype != np.float64 and dapi_rescaled.dtype != np.float32:
         # Handle integer types if rescale_intensity returned original dtype
         # Ensure arithmetic is done in float to prevent wrap-around
         b_channel = np.clip(dapi_rescaled.astype(float) + magenta_rescaled.astype(float), 0, np.iinfo(dapi.dtype).max)
         # Normalize to 0-1 for img_as_ubyte if necessary, but skimage handles float inputs best
    
    # Stack into (Height, Width, 3)
    rgb_composite = np.dstack((r_channel, g_channel, b_channel))

    # Convert to 8-bit unsigned integer for PNG saving
    rgb_uint8 = img_as_ubyte(rgb_composite)

    # Save output
    filename = os.path.join(output_dir, f"frame_{i:03d}.png")
    io.imsave(filename, rgb_uint8)
    
    print(f"Frame {i} saved to {filename}")

print("Processing complete.")

  0%|          | 0/27 [00:00<?, ?it/s]

Frame 0 saved to processed_frames/frame_000.png
Frame 1 saved to processed_frames/frame_001.png
Frame 2 saved to processed_frames/frame_002.png
Frame 3 saved to processed_frames/frame_003.png
Frame 4 saved to processed_frames/frame_004.png
Frame 5 saved to processed_frames/frame_005.png
Frame 6 saved to processed_frames/frame_006.png
Frame 7 saved to processed_frames/frame_007.png
Frame 8 saved to processed_frames/frame_008.png
Frame 9 saved to processed_frames/frame_009.png
Frame 10 saved to processed_frames/frame_010.png
Frame 11 saved to processed_frames/frame_011.png
Frame 12 saved to processed_frames/frame_012.png
Frame 13 saved to processed_frames/frame_013.png
Frame 14 saved to processed_frames/frame_014.png
Frame 15 saved to processed_frames/frame_015.png
Frame 16 saved to processed_frames/frame_016.png
Frame 17 saved to processed_frames/frame_017.png
Frame 18 saved to processed_frames/frame_018.png
Frame 19 saved to processed_frames/frame_019.png
Frame 20 saved to processed_fr

In [13]:
%%time
loaded_stack = stack.compute()

CPU times: user 2min 37s, sys: 51min 49s, total: 54min 26s
Wall time: 5min 59s


In [17]:
viewer = napari.Viewer()

In [18]:
viewer.add_image(
    loaded_stack, 
    channel_axis=1,
    # name='Mtb Stack Composite',
    colormap=['blue', 'green', 'magenta']
)

[<Image layer 'Image' at 0x7d3b9853e9e0>,
 <Image layer 'Image [1]' at 0x7d3ba837c3d0>,
 <Image layer 'Image [2]' at 0x7d3b98447be0>]

# Iteratively saving out zarr

In [18]:
from pathlib import Path
import zarr
import napari
from tqdm.auto import tqdm
from natsort import natsorted
import glob
import os
import numpy as np

In [73]:
fns = natsorted(glob.glob('/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep*/mouse_*/zarr/*.zarr'))
fns = [fn for fn in fns
        if any(sub in fn for sub in ['top', 'bot', 'left', 'right'])]

In [76]:
import os
import glob
import time
import zarr
import napari
import numpy as np
from tqdm import tqdm

# Ensure SCALE_VECTOR is defined in your global scope, e.g., (1, 0.5, 0.5)
output_dir = "/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/results/screenshots"
os.makedirs(output_dir, exist_ok=True)

viewer = napari.Viewer()

for fn in tqdm(fns):
    # Load Image Data
    zarr_group = zarr.open(fn, mode='r')
    pyramid_keys = sorted(
        [k for k in zarr_group.keys() if k.startswith('s') and k[1:].isdigit()],
        key=lambda x: int(x[1:])
    )
    pyramid_data = [zarr_group[k] for k in pyramid_keys]

    image_layer = viewer.add_image(
        pyramid_data,
        channel_axis=0,
        colormap=['blue', 'green', 'magenta'],
        name=['DAPI', 'Green', 'Red'],
        scale=SCALE_VECTOR if 'SCALE_VECTOR' in locals() else None 
    )

    # Load Masks
    mtb_layer = None
    mtb_mask_path = glob.glob(os.path.join(fn, 'labels/ground_truth_*/0'))
    if mtb_mask_path:
        mask_data = zarr.open(mtb_mask_path[0], mode='r')
        mtb_layer = viewer.add_labels(
            mask_data,
            scale=SCALE_VECTOR[-2:] if 'SCALE_VECTOR' in locals() else None,
            name='Mtb_Mask',
            visible=False
        )

    cyto_layer = None
    cyto_mask_path = os.path.join(fn, 'labels/qupath_cyto_segmentation/0')
    if os.path.exists(cyto_mask_path):
        mask_data = zarr.open(cyto_mask_path, mode='r')
        cyto_layer = viewer.add_labels(
            mask_data,
            scale=SCALE_VECTOR[-2:] if 'SCALE_VECTOR' in locals() else None,
            name='Cyto_Mask',
            visible=False
        )

    # Configure Scale Bar
    viewer.scale_bar.visible = True
    viewer.scale_bar.unit = 'um'
    viewer.scale_bar.position = 'bottom_right'

    # Define State Matrix
    # (Scale Name, Target FOV in um (approx), Zoom Logic)
    # Note: Napari scale bar adjusts to zoom. 1mm bar implies large FOV (~3-4mm). 50um bar implies small FOV (~150um).
    scales = [
        ('1mm_scale', 'reset'),  
        ('50um_scale', 150)       
    ]
    
    # (Condition Name, Cyto Visible, Mtb Visible)
    mask_states = [
        ('CytoOn_MtbOn', True, True),
        ('CytoOn_MtbOff', True, False),
        ('CytoOff_MtbOn', False, True),
        ('CytoOff_MtbOff', False, False)
    ]

    base_name = os.path.basename(fn)

    for scale_name, fov_param in scales:
        # Set Zoom
        if fov_param == 'reset':
            viewer.reset_view()
        else:
            # Calculate zoom: canvas_height / (target_fov / pixel_size)
            # Assuming pixel_size is roughly SCALE_VECTOR[-1] or 1 if undefined
            px_size = SCALE_VECTOR[-1] if 'SCALE_VECTOR' in locals() else 1.0
            canvas_height = viewer.window.qt_viewer.canvas.size[1]
            target_zoom = canvas_height / (fov_param / px_size)
            viewer.camera.zoom = target_zoom
            # Center camera if needed (optional, defaults to center of data)
            # viewer.camera.center = (z, y, x) 

        for condition_name, cyto_vis, mtb_vis in mask_states:
            if cyto_layer: cyto_layer.visible = cyto_vis
            if mtb_layer: mtb_layer.visible = mtb_vis
            
            # Allow rendering to catch up
            time.sleep(0.5) 
            
            save_path = f"{output_dir}/{base_name}_{scale_name}_{condition_name}.png"
            viewer.screenshot(save_path)

    # Clean up for next iteration to prevent memory leaks
    viewer.layers.clear()

viewer.close()

  0%|                                                                                                                           | 0/36 [00:00<?, ?it/s]/home/dayn/miniconda3/envs/macrohet_fixed/lib/python3.10/site-packages/napari/layers/utils/stack_utils.py:46: UserWarning: zarr array cannot be sliced lazily, converted to dask array.
  warnings.warn(
/home/dayn/miniconda3/envs/macrohet_fixed/lib/python3.10/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (35221, 44308) exceeds GL_MAX_TEXTURE_SIZE 16384 in at least one axis and will be downsampled. Rendering is currently in 2D mode.
  warnings.warn(
/home/dayn/miniconda3/envs/macrohet_fixed/lib/python3.10/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (54144, 45850) exceeds GL_MAX_TEXTURE_SIZE 16384 in at least one axis and will be downsampled. Rendering is currently in 2D mode.
  warnings.warn(
/tmp/ipykernel_1136422/1861555023.py:86: FutureWarning: Public access to Window.

In [75]:
# 2. Define the scale for the spatial dimensions (Y and X)
PIXEL_SIZE_UM = 0.165 
# 3. Define the full scale vector for all 5 dimensions (T, C, Z, Y, X)
SCALE_VECTOR = [PIXEL_SIZE_UM, PIXEL_SIZE_UM] 

In [71]:
viewer = napari.Viewer(title='generating balanced lung slice previews')

In [44]:
glob.glob(os.path.join(fn, 'labels/ground_truth_bot/*'))

['/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep1/mouse_6/zarr/rep1_mouse6_bot.zarr/labels/ground_truth_bot/0']

In [50]:
zarr_group

<zarr.core.Array (3, 35221, 44308) >u2 read-only>

In [55]:
zarr_group = zarr.open(f'{fn}', mode='r')
# Extract and sort keys that follow the 's0', 's1' pattern
pyramid_keys = sorted(
    [k for k in zarr_group.keys() if k.startswith('s') and k[1:].isdigit()],
    key=lambda x: int(x[1:])
)
pyramid_data = [zarr_group[k] for k in pyramid_keys]


In [67]:
pyramid_data

[<zarr.core.Array '/0' (1, 3, 11, 31334, 49997) >u2 read-only>,
 <zarr.core.Array '/1' (1, 3, 11, 15667, 24998) >u2 read-only>,
 <zarr.core.Array '/2' (1, 3, 11, 7833, 12499) >u2 read-only>,
 <zarr.core.Array '/3' (1, 3, 11, 3916, 6249) >u2 read-only>,
 <zarr.core.Array '/4' (1, 3, 11, 1958, 3124) >u2 read-only>,
 <zarr.core.Array '/5' (1, 3, 11, 979, 1562) >u2 read-only>,
 <zarr.core.Array '/6' (1, 3, 11, 489, 781) >u2 read-only>,
 <zarr.core.Array '/7' (1, 3, 11, 244, 390) >u2 read-only>,
 <zarr.core.Array '/8' (1, 3, 11, 122, 195) >u2 read-only>]

In [69]:
fn

'/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep1/mouse_1/zarr/20250815_40X_TimerMtb_BP_mice1_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250815_5375.zarr'

In [72]:
for fn in tqdm(fns):
    # zarr_group = zarr.open(f'{fn}/0', mode='r')
    # pyramid_data = [zarr_group[k] for k in sorted(zarr_group.keys(), key=int) if k.isdigit()]
    Apply the projection to each level of the pyramid
    # pyramid_data = [np.max(level, axis=2) for level in pyramid_data]
    zarr_group = zarr.open(f'{fn}', mode='r')
    # Extract and sort keys that follow the 's0', 's1' pattern
    pyramid_keys = sorted(
        [k for k in zarr_group.keys() if k.startswith('s') and k[1:].isdigit()],
        key=lambda x: int(x[1:])
    )
    pyramid_data = [zarr_group[k] for k in pyramid_keys]

    
    viewer.add_image(
        pyramid_data,
        # scale=SCALE_VECTOR,
        # multiscale=True,
        # units='micrometer',
        channel_axis=1,
        colormap=['blue', 'green', 'magenta'],
        name=['DAPI', 'Green', 'Red']
    )

    mtb_mask_path = glob.glob(os.path.join(fn, 'labels/ground_truth_*/0'))[0]
    if os.path.exists(mtb_mask_path):
        mask_data = zarr.open(mtb_mask_path, mode='r')
        viewer.add_labels(
            mask_data,
            scale=SCALE_VECTOR[-2:],
            name='Mtb_Mask'
        )
    cyto_mask_path = os.path.join(fn, 'labels/qupath_cyto_segmentation/0')
    if os.path.exists(cyto_mask_path):
        mask_data = zarr.open(cyto_mask_path, mode='r')
        viewer.add_labels(
            mask_data,
            scale=SCALE_VECTOR[-2:],
            name='Cyto_Mask'
        )
    viewer.scale_bar.visible = True
    viewer.scale_bar.unit = 'um'
    viewer.scale_bar.position = 'bottom_right'
    break
    # viewer.show(block=True)

  0%|          | 0/27 [00:00<?, ?it/s]

/home/dayn/miniconda3/envs/macrohet_fixed/lib/python3.10/site-packages/napari/layers/utils/stack_utils.py:46: UserWarning: zarr array cannot be sliced lazily, converted to dask array.
  warnings.warn(
/home/dayn/miniconda3/envs/macrohet_fixed/lib/python3.10/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (31334, 49997) exceeds GL_MAX_TEXTURE_SIZE 16384 in at least one axis and will be downsampled. Rendering is currently in 2D mode.
  warnings.warn(
/home/dayn/miniconda3/envs/macrohet_fixed/lib/python3.10/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (31334, 49997) exceeds GL_MAX_TEXTURE_SIZE 16384 in at least one axis and will be downsampled. Rendering is currently in 2D mode.
  warnings.warn(


In [32]:
mask_path

'/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/data/whole_slice/rep1/mouse_1/zarr/20250815_40X_TimerMtb_BP_mice1_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250815_5375.zarr/labels/ground_truth_mtb/0'